<a href="https://colab.research.google.com/github/chadallen/insider_trading_detection/blob/main/Detect_insider_trading_on_prediction_markets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook looks for suspicious trading patterns in resolved Polymarket prediction markets.

**How it works**
1. We pull ~120 recently resolved markets and score each one across four signals:

2. VPIN — Models for detecting insider trading in
*financial markets* (Easley, López de Prado & O'Hara, 2011, 2012)
3. Time-weighted VPIN — same, but weighted toward moves closer to resolution
4. Resolution surprise — how wrong was the market's implied probability vs. what actually happened?
5. Late move ratio — what % of total price movement happened right before resolution?
6. We then run Isolation Forest across all four signals to rank markets by how anomalous they look.

Note: Polymarket only gives us 12h price granularity for closed markets, so VPIN is approximated from price movement rather than actual order flow.
No labeled training data yet — this is unsupervised
~120 markets is a small sample



In [2]:
#@title 1 - Install dependencies

!pip install requests pandas numpy scikit-learn plotly -q
!pip install PyGithub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.7/432.7 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.5 MB/s eta 0:00:00


In [3]:
#@title 1b - Mount Google Drive and load saved data

from google.colab import drive
import os
import pickle

print(os.listdir("/content/drive/MyDrive"))
print(os.listdir("/content/drive/MyDrive/insider_trading_poc"))

try:
    drive.mount('/content/drive')
except ValueError:
    print("Drive already mounted ✅")


SAVE_DIR = "/content/drive/MyDrive/insider_trading_poc"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")

def load_checkpoint(name):
    path = f"{SAVE_DIR}/{name}.pkl"
    if os.path.exists(path):
        with open(path, "rb") as f:
            data = pickle.load(f)
        print(f"  ✅ Loaded {name}")
        return data
    else:
        print(f"  ⚠️  Not found: {name} (will be computed fresh)")
        return None

print("\nLoading saved data...\n")

dune_results  = load_checkpoint("dune_results")
df_scored     = load_checkpoint("df_scored")
df_batch      = load_checkpoint("df_batch")
df_combined   = load_checkpoint("df_combined")
df_markets    = load_checkpoint("df_markets")
histories     = load_checkpoint("histories")
df_wallet_agg = load_checkpoint("df_wallet_agg")

print("\nDone. Any None values will be recomputed when you run their cells.")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive'

In [4]:
#@title 2 - Fetch resolved markets

import requests
import pandas as pd
import json
from datetime import datetime, timedelta, timezone

def fetch_resolved_markets(days_back=365, limit=200):
    """Fetch recently resolved markets with token IDs included."""

    end_date = datetime.now(timezone.utc)
    start_date = end_date - timedelta(days=days_back)

    url = "https://gamma-api.polymarket.com/markets"
    params = {
        "closed": "true",
        "limit": limit,
        "after": start_date.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "order": "endDate",
        "ascending": "false",
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    markets = []
    for m in data:
        # parse token IDs
        raw_tokens = m.get("clobTokenIds", "[]")
        try:
            tokens = json.loads(raw_tokens) if isinstance(raw_tokens, str) else raw_tokens
            token_id = tokens[0] if tokens else None
        except:
            token_id = None

        markets.append({
            "market_id":       m.get("conditionId"),
            "token_id":        token_id,
            "question":        m.get("question"),
            "resolution_time": m.get("endDate"),
            "final_price":     m.get("outcomePrices"),
            "volume":          float(m.get("volume") or 0),
        })

    df = pd.DataFrame(markets)
    # drop rows with no token_id or no volume
    df = df[df["token_id"].notna() & (df["volume"] > 0)]
    print(f"Fetched {len(df)} markets with trade data")
    return df

df_markets = fetch_resolved_markets(days_back=548)
print(df_markets[["question", "volume", "token_id"]].head(5))

Fetched 181 markets with trade data
                                          question         volume  \
1   Espresso FDV above $700M one day after launch?   42124.516329   
5    Espresso FDV above $50M one day after launch?   83280.457952   
7   Espresso FDV above $400M one day after launch?   90657.051213   
8   Espresso FDV above $300M one day after launch?  140357.762760   
11  Espresso FDV above $100M one day after launch?  211099.127702   

                                             token_id  
1   7499602479293181980852689956064562479832303792...  
5   9626745825409044179824523696953147620278933733...  
7   7856813241556718272610467770982419627408670319...  
8   9520754463524584396923550946076907828609682174...  
11  3677484426870441719637920109352048482658621079...  


In [5]:
#@title 3 - Fetch price histories

def fetch_price_history(token_id, resolution_time, hours_before=48):
    """Fetch price history - note: closed markets only support 12h+ granularity."""

    if isinstance(resolution_time, str):
        res_time = datetime.fromisoformat(resolution_time.replace("Z", "+00:00"))
    else:
        res_time = resolution_time

    start_time = res_time - timedelta(hours=hours_before)

    # correct host: clob not gamma-api
    url = "https://clob.polymarket.com/prices-history"
    params = {
        "market":    token_id,
        "interval":  "max",
        "fidelity":  720,   # 12 hours in minutes - minimum for closed markets
        "startTs":   int(start_time.timestamp()),
        "endTs":     int(res_time.timestamp()),
    }

    try:
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        data = r.json()

        if not data or "history" not in data or not data["history"]:
            return pd.DataFrame()

        # columns are 't' and 'p' not 'timestamp' and 'price'
        df = pd.DataFrame(data["history"])
        df = df.rename(columns={"t": "timestamp", "p": "price"})
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)
        df["price"] = df["price"].astype(float)
        return df

    except Exception as e:
        print(f"  Error: {e}")
        return pd.DataFrame()

# Test on first 5 markets
for _, row in df_markets.head(5).iterrows():
    history = fetch_price_history(row["token_id"], row["resolution_time"])
    print(f"Market: {row['question'][:55]}...")
    print(f"  Points: {len(history)}  |  Volume: ${row['volume']:,.0f}")
    if len(history) > 0:
        print(f"  Price: {history['price'].iloc[0]:.2f} → {history['price'].iloc[-1]:.2f}")
    print()

Market: Espresso FDV above $700M one day after launch?...
  Points: 6  |  Volume: $42,125
  Price: 0.02 → 0.00

Market: Espresso FDV above $50M one day after launch?...
  Points: 8  |  Volume: $83,280
  Price: 0.99 → 1.00

Market: Espresso FDV above $400M one day after launch?...
  Points: 8  |  Volume: $90,657
  Price: 0.27 → 0.00

Market: Espresso FDV above $300M one day after launch?...
  Points: 8  |  Volume: $140,358
  Price: 0.48 → 0.00

Market: Espresso FDV above $100M one day after launch?...
  Points: 8  |  Volume: $211,099
  Price: 0.97 → 1.00



In [6]:
#@title 4 - Compute VPIN and price features

import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

def compute_vpin_from_prices(history_df):
    if len(history_df) < 3:
        return None
    prices = history_df["price"].values
    changes = np.diff(prices)
    buy_pressure = np.where(changes > 0, changes, 0).sum()
    sell_pressure = np.where(changes < 0, -changes, 0).sum()
    total = buy_pressure + sell_pressure
    if total == 0:
        return 0.0
    return abs(buy_pressure - sell_pressure) / total

def compute_price_features(history_df):
    if len(history_df) < 2:
        return None
    prices = history_df["price"].values
    changes = np.abs(np.diff(prices))
    return {
        "price_volatility": changes.std() if len(changes) > 1 else 0,
        "max_single_move": changes.max() if len(changes) > 0 else 0,
        "final_price": prices[-1],
        "starting_price": prices[0],
        "total_price_move": abs(prices[-1] - prices[0]),
    }

histories = {}  # NEW — cache for Cell 4b

results = []
for _, row in df_markets.iterrows():
    history = fetch_price_history(row["token_id"], row["resolution_time"])
    if len(history) < 3:
        continue

    starting_price = history["price"].iloc[0]
    if not (0.15 <= starting_price <= 0.85):
        continue

    histories[row["token_id"]] = history  # NEW — store history for reuse

    vpin = compute_vpin_from_prices(history)
    features = compute_price_features(history)
    if vpin is None or features is None:
        continue

    results.append({
        "question": row["question"],
        "volume": row["volume"],
        "vpin_score": vpin,
        **features,
    })

df_scored = pd.DataFrame(results)
print(f"Markets with enough data: {len(df_scored)}")

feature_cols = ["vpin_score", "volume", "total_price_move", "price_volatility", "max_single_move"]
X = df_scored[feature_cols].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
model = IsolationForest(contamination=0.1, random_state=42)
df_scored["anomaly_score"] = model.fit_predict(X_scaled)
df_scored["suspicion_score"] = -model.decision_function(X_scaled)

top = df_scored.nlargest(15, "suspicion_score")[
    ["question", "suspicion_score", "vpin_score", "volume", "starting_price", "final_price"]
].reset_index(drop=True)
print(top.to_string())

Markets with enough data: 124
                                             question  suspicion_score  vpin_score        volume  starting_price  final_price
0   Will Almanak FDV above $50M one day after launch?         0.146778    0.997650  2.053399e+05           0.850       0.0010
1      HumidiFi FDV above $300M one day after launch?         0.097377    0.149594  2.296843e+05           0.200       0.0065
2         Owlto FDV above $200M one day after launch?         0.083490    0.588477  4.267078e+04           0.205       0.9915
3   Will Bitcoin dip to $65,000 by December 31, 2026?         0.074854    0.090032  1.535035e+06           0.590       0.8700
4           Trove FDV above $5M one day after launch?         0.051637    1.000000  2.347774e+05           0.815       0.0005
5       Kinetiq FDV above $250M one day after launch?         0.029769    0.317525  8.300058e+05           0.825       0.0015
6      Over $2M committed to the Infinex public sale?         0.029701    0.990878  7.31

In [8]:
#@title 5 - Compute resolution surprise features

def compute_resolution_surprise(history_df):
    if len(history_df) < 2:
        return None, None
    prices = history_df["price"].values
    actual_resolution = 1.0 if prices[-1] > 0.5 else 0.0
    prior_price = prices[0]
    surprise_score = abs(actual_resolution - prior_price)
    total_move = abs(prices[-1] - prices[0])
    final_step_move = abs(prices[-1] - prices[-2])
    late_move_ratio = (final_step_move / total_move) if total_move > 0.01 else 0.0
    return surprise_score, late_move_ratio

surprise_scores = []
late_move_ratios = []

for _, row in df_scored.iterrows():
    token_id = df_markets.loc[
        df_markets["question"] == row["question"], "token_id"
    ].values[0]
    history = histories.get(token_id)
    surprise, late_move = compute_resolution_surprise(history) if history is not None else (0.0, 0.0)
    surprise_scores.append(surprise)
    late_move_ratios.append(late_move)

df_scored["surprise_score"] = surprise_scores
df_scored["late_move_ratio"] = late_move_ratios

print(df_scored[["question", "surprise_score", "late_move_ratio"]].head(10).to_string())

                                             question  surprise_score  late_move_ratio
0      Espresso FDV above $400M one day after launch?           0.265         0.011342
1      Espresso FDV above $300M one day after launch?           0.480         0.013556
2      Espresso FDV above $200M one day after launch?           0.550         0.845314
3          VOOI FDV above $800M one day after launch?           0.265         0.000000
4         Trove FDV above $100M one day after launch?           0.155         0.000000
5   Will Almanak FDV above $50M one day after launch?           0.850         0.000589
6   Over $5M committed to the Sol Raffle public sale?           0.185         0.005420
7     Over $20M committed to the Infinex public sale?           0.425         0.009756
8  Will the Ethereum Volatility Index hit 90 in 2026?           0.335         0.076923
9   Will ETHGAS launch a token by September 30, 2026?           0.200         0.005013


In [9]:
#@title  6 - Compute time-weighted VPIN

def compute_time_weighted_vpin(history_df):
    """
    Like VPIN, but movements closer to resolution are weighted more heavily.
    Weight increases linearly: first move = 1, last move = N (number of steps).
    """
    if len(history_df) < 3:
        return None

    prices = history_df["price"].values
    changes = np.diff(prices)
    n = len(changes)

    # Linear weights: 1, 2, 3 ... N (last move gets highest weight)
    weights = np.arange(1, n + 1, dtype=float)

    weighted_buy = np.where(changes > 0, changes * weights, 0).sum()
    weighted_sell = np.where(changes < 0, -changes * weights, 0).sum()
    total = weighted_buy + weighted_sell

    if total == 0:
        return 0.0
    return abs(weighted_buy - weighted_sell) / total

# Add to df_scored
tw_vpins = []
for _, row in df_scored.iterrows():
    token_id = df_markets.loc[
        df_markets["question"] == row["question"], "token_id"
    ].values[0]
    history = histories.get(token_id)
    tw_vpin = compute_time_weighted_vpin(history) if history is not None else 0.0
    tw_vpins.append(tw_vpin)

df_scored["time_weighted_vpin"] = tw_vpins
print(df_scored[["question", "vpin_score", "time_weighted_vpin"]].head(10).to_string())

                                             question  vpin_score  time_weighted_vpin
0      Espresso FDV above $400M one day after launch?    0.725652            0.690243
1      Espresso FDV above $300M one day after launch?    0.905571            0.918853
2      Espresso FDV above $200M one day after launch?    0.465875            0.435484
3          VOOI FDV above $800M one day after launch?    0.684347            0.405771
4         Trove FDV above $100M one day after launch?    0.378213            0.211188
5   Will Almanak FDV above $50M one day after launch?    0.997650            0.994889
6   Over $5M committed to the Sol Raffle public sale?    0.994609            0.991935
7     Over $20M committed to the Infinex public sale?    0.636646            0.283884
8  Will the Ethereum Volatility Index hit 90 in 2026?    0.276596            0.112903
9   Will ETHGAS launch a token by September 30, 2026?    0.134389            0.253437


In [10]:
#@title 7 - Score markets with Isolation Forest

feature_cols = [
    "vpin_score", "time_weighted_vpin",  # ← add this
    "volume", "total_price_move",
    "price_volatility", "max_single_move",
    "surprise_score", "late_move_ratio"
]
X = df_scored[feature_cols].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
model = IsolationForest(contamination=0.1, random_state=42)
df_scored["anomaly_score"] = model.fit_predict(X_scaled)
df_scored["suspicion_score"] = -model.decision_function(X_scaled)

top = df_scored.nlargest(15, "suspicion_score")[
    ["question", "suspicion_score", "vpin_score", "surprise_score",
     "late_move_ratio", "volume", "starting_price", "final_price"]
].reset_index(drop=True)
print(top.to_string())


                                             question  suspicion_score  vpin_score  surprise_score  late_move_ratio        volume  starting_price  final_price
0      HumidiFi FDV above $300M one day after launch?         0.150967    0.149594           0.200         3.041344  2.296843e+05           0.200       0.0065
1   Will Almanak FDV above $50M one day after launch?         0.123843    0.997650           0.850         0.000589  2.053399e+05           0.850       0.0010
2         Owlto FDV above $200M one day after launch?         0.085602    0.588477           0.795         0.716465  4.267078e+04           0.205       0.9915
3           Trove FDV above $5M one day after launch?         0.072537    1.000000           0.815         0.079190  2.347774e+05           0.815       0.0005
4   Will the Bitcoin Volatility Index hit 70 in 2026?         0.067744    0.022989           0.470         2.250000  4.216766e+03           0.470       0.4900
5   Will Bitcoin dip to $65,000 by December 31

In [ ]:
#@title 8 - deprecated


In [18]:
#@title 9 - Preview results

print(df_scored.nlargest(30, "suspicion_score").to_csv(index=False))

question,volume,vpin_score,price_volatility,max_single_move,final_price,starting_price,total_price_move,anomaly_score,suspicion_score,surprise_score,late_move_ratio,time_weighted_vpin
Stable FDV above $2B one day after launch?,2108189.721855,0.26884729753367154,0.06717622875598588,0.56,0.0015,0.77,0.7685000000000001,-1,0.13183589654193706,0.77,0.22576447625243978,0.45790710972742876
Will Almanak FDV above $50M one day after launch?,205339.945813,0.9976498237367802,0.21113042309329924,0.59,0.001,0.85,0.849,-1,0.10803877872663548,0.85,0.0005889281507656067,0.9948888320981345
HumidiFi FDV above $300M one day after launch?,229684.290415,0.14959412446849632,0.19289375313887178,0.5885,0.0065,0.2,0.1935,-1,0.10742248143195243,0.2,3.041343669250646,0.14821235811716985
Sentient FDV above $1B one day after launch?,522126.144342,0.010550996483001259,0.04782078091585649,0.355,0.44,0.395,0.044999999999999984,-1,0.10283879337522384,0.395,7.888888888888891,0.02462513923399881
Sentient FDV above $800M

In [19]:
#@title 10 - Load labeled insider trading cases

import io

LABELS_CSV = """market_question,label
"Will the US strike Iran by February 28, 2026?",SUSPECTED
"Khamenei out as Supreme Leader of Iran by March 31, 2026?",SUSPECTED
"Will Nicolás Maduro be removed from office by January 31, 2026?",SUSPECTED
"Will the US take military action against Venezuela by January 31, 2026?",SUSPECTED
"Will María Corina Machado win the 2025 Nobel Peace Prize?",CONFIRMED
"Will Israel strike Iran between June 12–18, 2025?",CONFIRMED
"Which crypto company will ZachXBT expose for insider trading?",SUSPECTED
"Will Taylor Swift and Travis Kelce get engaged in 2025?",SUSPECTED
"Who will be the most searched person on Google in 2025?",SUSPECTED
"Will Google release Gemini 3.0 Flash on specific date window?",POSSIBLE
"Will the 2024 US presidential election be won by Donald Trump?",POSSIBLE
"""

df_labels = pd.read_csv(io.StringIO(LABELS_CSV))
print(f"Loaded {len(df_labels)} labeled cases")
print(df_labels[["market_question", "label"]].to_string())

Loaded 11 labeled cases
                                                            market_question      label
0                             Will the US strike Iran by February 28, 2026?  SUSPECTED
1                 Khamenei out as Supreme Leader of Iran by March 31, 2026?  SUSPECTED
2           Will Nicolás Maduro be removed from office by January 31, 2026?  SUSPECTED
3   Will the US take military action against Venezuela by January 31, 2026?  SUSPECTED
4                 Will María Corina Machado win the 2025 Nobel Peace Prize?  CONFIRMED
5                         Will Israel strike Iran between June 12–18, 2025?  CONFIRMED
6             Which crypto company will ZachXBT expose for insider trading?  SUSPECTED
7                   Will Taylor Swift and Travis Kelce get engaged in 2025?  SUSPECTED
8                   Who will be the most searched person on Google in 2025?  SUSPECTED
9             Will Google release Gemini 3.0 Flash on specific date window?   POSSIBLE
10           Will t


Note: Price model cannot score labeled markets

We attempted to score the labeled insider trading cases through the price model
(Isolation Forest) using CLOB price history. All 11 labeled markets returned
NO HISTORY — the Polymarket CLOB API does not retain price data for closed
political markets beyond a short window after resolution.

This means the price model and the labeled cases are scoring completely different
universes:
- Price model: ~132 resolved crypto token launch markets (FDV markets)
- Labeled cases: political markets (Iran, Nobel, Maduro, ZachXBT)

The validation path for labeled cases is wallet-based only — see Cells 11 and 12.
Fixing this gap (by reconstructing price history from Dune trade data) is a
priority next step — see project plan.

In [13]:
#@title 11 - Fetch labeled market trades from Dune API

import requests
import time
from google.colab import userdata

DUNE_API_KEY = userdata.get('DUNE_API_KEY')

HEADERS = {
    "X-Dune-Api-Key": DUNE_API_KEY,
    "Content-Type": "application/json"
}

def execute_sql(sql):
    r = requests.post(
        "https://api.dune.com/api/v1/sql/execute",
        headers=HEADERS,
        json={"sql": sql, "performance": "medium"}
    )
    r.raise_for_status()
    return r.json()["execution_id"]

def poll_until_done(execution_id, timeout=120, interval=5):
    start = time.time()
    while time.time() - start < timeout:
        r = requests.get(
            f"https://api.dune.com/api/v1/execution/{execution_id}/status",
            headers=HEADERS
        )
        r.raise_for_status()
        state = r.json()["state"]
        if state == "QUERY_STATE_COMPLETED":
            return True
        elif state in ("QUERY_STATE_FAILED", "QUERY_STATE_CANCELLED"):
            print(f"  Query failed: {state}")
            return False
        print(f"  Status: {state} — waiting {interval}s...")
        time.sleep(interval)
    print("  Timed out.")
    return False

def fetch_results(execution_id):
    r = requests.get(
        f"https://api.dune.com/api/v1/execution/{execution_id}/results",
        headers=HEADERS
    )
    r.raise_for_status()
    rows = r.json().get("result", {}).get("rows", [])
    return pd.DataFrame(rows)

QUERIES = {
    "maduro": {
        "label": "SUSPECTED",
        "sql": """
            SELECT block_time, question, asset_id, price, amount, shares, maker, taker, token_outcome_name
            FROM polymarket_polygon.market_trades
            WHERE question LIKE '%Maduro%'
              AND block_time BETWEEN TIMESTAMP '2026-01-28' AND TIMESTAMP '2026-01-31'
            ORDER BY block_time ASC
        """
    },
    "zachxbt": {
        "label": "SUSPECTED",
        "sql": """
            SELECT block_time, question, asset_id, price, amount, shares, maker, taker, token_outcome_name
            FROM polymarket_polygon.market_trades
            WHERE question = 'Will Axiom be accused of insider trading?'
              AND block_time BETWEEN TIMESTAMP '2026-02-24' AND TIMESTAMP '2026-02-27'
            ORDER BY block_time ASC
        """
    },
    "nobel": {
        "label": "CONFIRMED",
        "sql": """
            SELECT block_time, question, asset_id, price, amount, shares, maker, taker, token_outcome_name
            FROM polymarket_polygon.market_trades
            WHERE question = 'Will María Corina Machado win the Nobel Peace Prize in 2025?'
              AND block_time BETWEEN TIMESTAMP '2025-10-08' AND TIMESTAMP '2025-10-11'
            ORDER BY block_time ASC
        """
    },
    "iran": {
        "label": "CONFIRMED",
        "sql": """
            SELECT block_time, question, asset_id, price, amount, shares, maker, taker, token_outcome_name
            FROM polymarket_polygon.market_trades
            WHERE question = 'US strikes Iran by February 28, 2026?'
              AND block_time BETWEEN TIMESTAMP '2026-02-26' AND TIMESTAMP '2026-02-28'
            ORDER BY block_time ASC
        """
    },
}

dune_results = {}

for name, config in QUERIES.items():
    print(f"\n── {name.upper()} ({config['label']}) ──────────────────────────────────")
    try:
        exec_id = execute_sql(config["sql"])
        print(f"  Execution ID: {exec_id}")
        success = poll_until_done(exec_id)
        if success:
            df = fetch_results(exec_id)
            dune_results[name] = df
            print(f"  ✅ Got {len(df)} rows")
            if len(df) > 0:
                print(f"  Time range: {df['block_time'].min()} → {df['block_time'].max()}")
                print(f"  Unique wallets: {pd.concat([df['maker'], df['taker']]).nunique()}")
        else:
            dune_results[name] = pd.DataFrame()
    except Exception as e:
        print(f"  ❌ Error: {e}")
        dune_results[name] = pd.DataFrame()

print("\n── Summary ────────────────────────────────────────────────────────────")
for name, df in dune_results.items():
    print(f"  {name}: {len(df)} rows")


── MADURO (SUSPECTED) ──────────────────────────────────
  Execution ID: 01KK7YKQSG2SFTVFWY9MAHKM3Z
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PEN

In [16]:
#@title 12 - Compute wallet-level features for labeled markets

def compute_wallet_features(df, resolution_time_str):
    """
    Compute wallet-level features from raw trade data.

    Args:
        df: DataFrame with columns block_time, maker, taker, amount, price, token_outcome_name
        resolution_time_str: ISO string of market resolution time
    """
    if len(df) == 0:
        return None

    # Parse times
    df = df.copy()
    df["block_time"] = pd.to_datetime(df["block_time"], utc=True)
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce").fillna(0)
    df["price"] = pd.to_numeric(df["price"], errors="coerce").fillna(0)

    resolution_time = pd.Timestamp(resolution_time_str, tz="UTC")
    final_12h_start = resolution_time - pd.Timedelta(hours=12)

    # Use maker as the primary wallet identifier
    df["wallet"] = df["maker"]
    total_volume = df["amount"].sum()

    # ── Feature 1: Wallet concentration ─────────────────────────────────────
    wallet_volume = df.groupby("wallet")["amount"].sum().sort_values(ascending=False)
    top3_volume = wallet_volume.head(3).sum()
    wallet_concentration = top3_volume / total_volume if total_volume > 0 else 0

    # ── Feature 2: New wallet ratio ──────────────────────────────────────────
    # "New" = wallet whose first trade in our window is in the final 12h
    wallet_first_trade = df.groupby("wallet")["block_time"].min()
    new_wallets = wallet_first_trade[wallet_first_trade >= final_12h_start].index
    new_wallet_volume = df[df["wallet"].isin(new_wallets)]["amount"].sum()
    new_wallet_ratio = new_wallet_volume / total_volume if total_volume > 0 else 0

    # ── Feature 2b: New wallet ratio (final 6h) ──────────────────────────────
    final_6h_start = resolution_time - pd.Timedelta(hours=6)
    new_wallets_6h = wallet_first_trade[wallet_first_trade >= final_6h_start].index
    new_wallet_volume_6h = df[df["wallet"].isin(new_wallets_6h)]["amount"].sum()
    new_wallet_ratio_6h = new_wallet_volume_6h / total_volume if total_volume > 0 else 0

    # ── Feature 3: Burst trading score ──────────────────────────────────────
    # Max trades any single wallet placed in a 1-hour window
    df["hour_bucket"] = df["block_time"].dt.floor("h")
    trades_per_wallet_hour = df.groupby(["wallet", "hour_bucket"]).size()
    burst_score = trades_per_wallet_hour.max() if len(trades_per_wallet_hour) > 0 else 0

    # ── Feature 4: Directional consensus ────────────────────────────────────
    # % of volume on the winning outcome
    outcome_volumes = df.groupby("token_outcome_name")["amount"].sum()
    if len(outcome_volumes) > 0:
        winning_outcome_volume = outcome_volumes.max()
        directional_consensus = winning_outcome_volume / total_volume if total_volume > 0 else 0
    else:
        directional_consensus = 0

    # ── Feature 5: Trade-direction VPIN ─────────────────────────────────────
    # Use price relative to 0.5 to classify buy/sell pressure
    # Trades above 0.5 = buying YES (bullish), below 0.5 = buying NO (bearish)
    df_sorted = df.sort_values("block_time")
    prices = df_sorted["price"].values
    amounts = df_sorted["amount"].values

    buy_volume = amounts[prices > 0.5].sum()
    sell_volume = amounts[prices <= 0.5].sum()
    total_directional = buy_volume + sell_volume

    trade_vpin = abs(buy_volume - sell_volume) / total_directional if total_directional > 0 else 0

    return {
        "total_volume": total_volume,
        "unique_wallets": df["wallet"].nunique(),
        "wallet_concentration": wallet_concentration,
        "new_wallet_ratio": new_wallet_ratio,
        "burst_score": int(burst_score),
        "directional_consensus": directional_consensus,
        "trade_vpin": trade_vpin,
        "top3_wallets": list(wallet_volume.head(3).index),
        "top3_volume_usd": round(top3_volume, 2),
        "new_wallet_ratio_6h": new_wallet_ratio_6h
    }


# Resolution times for each market
RESOLUTION_TIMES = {
    "maduro":  "2026-01-31T00:00:00",
    "zachxbt": "2026-02-26T23:59:00",
    "nobel":   "2025-10-10T11:00:00",
    "iran": "2026-02-28T23:59:00"

}

# Compute features for each market
print("── Wallet-level features ────────────────────────────────────────\n")
wallet_features = {}

for name, df in dune_results.items():
    if len(df) == 0:
        print(f"⚠️  {name}: no data\n")
        continue

    label = {"maduro": "SUSPECTED", "zachxbt": "SUSPECTED", "nobel": "CONFIRMED", "iran": "CONFIRMED"}[name]
    features = compute_wallet_features(df, RESOLUTION_TIMES[name])

    if features is None:
        continue

    wallet_features[name] = features

    print(f"── {name.upper()} ({label}) ──────────────────────────────")
    print(f"  Total volume:          ${features['total_volume']:,.2f}")
    print(f"  Unique wallets:        {features['unique_wallets']:,}")
    print(f"  Wallet concentration:  {features['wallet_concentration']:.1%}  (top 3 = {features['wallet_concentration']*100:.1f}% of volume)")
    print(f"  New wallet ratio:      {features['new_wallet_ratio']:.1%}  (% volume from wallets first seen in final 12h)")
    print(f"  Burst score:           {features['burst_score']}  (max trades by one wallet in 1h)")
    print(f"  Directional consensus: {features['directional_consensus']:.1%}  (% volume on winning side)")
    print(f"  Trade VPIN:            {features['trade_vpin']:.3f}")
    print(f"  Top 3 wallets:         {features['top3_wallets'][0][:12]}... (${features['top3_volume_usd']:,.0f} combined)")
    print(f"  New wallet ratio (6h): {features['new_wallet_ratio_6h']:.1%}  (% volume from wallets first seen in final 6h)")
    print()

── Wallet-level features ────────────────────────────────────────

⚠️  maduro: no data

⚠️  zachxbt: no data

⚠️  nobel: no data

── IRAN (CONFIRMED) ──────────────────────────────
  Total volume:          $13,092,871.72
  Unique wallets:        6,417
  Wallet concentration:  9.2%  (top 3 = 9.2% of volume)
  New wallet ratio:      0.0%  (% volume from wallets first seen in final 12h)
  Burst score:           409  (max trades by one wallet in 1h)
  Directional consensus: 77.6%  (% volume on winning side)
  Trade VPIN:            0.552
  Top 3 wallets:         0x96489abcb9... ($1,202,695 combined)
  New wallet ratio (6h): 0.0%  (% volume from wallets first seen in final 6h)



In [17]:
#@title 13 - Integrate wallet features into suspicion scoring

# ── 1. Define wallet suspicion thresholds ────────────────────────────────────
# Based on what we observed across our 4 labeled markets

def compute_wallet_suspicion_score(features):
    """
    Rule-based wallet suspicion score (0-1).
    Each component contributes equally (0.2 each).
    Thresholds derived from labeled market analysis.
    """
    score = 0.0

    # New wallet ratio 12h > 10% is elevated, > 50% is very suspicious
    nwr = features["new_wallet_ratio"]
    if nwr > 0.50:
        score += 0.20
    elif nwr > 0.10:
        score += 0.10

    # New wallet ratio 6h > 5% is notable
    nwr_6h = features["new_wallet_ratio_6h"]
    if nwr_6h > 0.10:
        score += 0.20
    elif nwr_6h > 0.05:
        score += 0.10

    # Trade VPIN > 0.7 is highly one-directional
    vpin = features["trade_vpin"]
    if vpin > 0.80:
        score += 0.20
    elif vpin > 0.60:
        score += 0.10

    # Burst score > 200 suggests algorithmic/bot behavior
    burst = features["burst_score"]
    if burst > 200:
        score += 0.20
    elif burst > 50:
        score += 0.10

    # Directional consensus > 70% means market was very one-sided
    dc = features["directional_consensus"]
    if dc > 0.70:
        score += 0.20
    elif dc > 0.55:
        score += 0.10

    return round(score, 2)


# ── 2. Score labeled markets ──────────────────────────────────────────────────
labels = {
    "maduro": "SUSPECTED",
    "zachxbt": "SUSPECTED",
    "nobel": "CONFIRMED",
    "iran": "CONFIRMED",
}

print("── Wallet suspicion scores (labeled markets) ────────────────────\n")
wallet_scores = []

for name, features in wallet_features.items():
    ws = compute_wallet_suspicion_score(features)
    wallet_scores.append({
        "market": name,
        "label": labels[name],
        "wallet_suspicion": ws,
        "new_wallet_12h": features["new_wallet_ratio"],
        "new_wallet_6h": features["new_wallet_ratio_6h"],
        "trade_vpin": features["trade_vpin"],
        "burst_score": features["burst_score"],
        "directional_consensus": features["directional_consensus"],
    })
    print(f"  {name.upper()} ({labels[name]}): wallet suspicion = {ws:.2f}")

df_wallet_scores = pd.DataFrame(wallet_scores)

# ── 3. Compare to price-based scores for overlapping markets ─────────────────
# Check if any of our labeled markets appear in df_scored
print("\n── Cross-checking against price-based model ─────────────────────\n")

market_name_map = {
    "maduro":  "Maduro",
    "zachxbt": "Axiom",
    "nobel":   "Machado",
    "iran":    "US strikes Iran",
}

for name in wallet_features:
    keyword = market_name_map[name]
    matches = df_scored[df_scored["question"].str.contains(keyword, case=False, na=False)]
    if len(matches) > 0:
        print(f"  {name.upper()} found in df_scored:")
        print(matches[["question", "suspicion_score"]].to_string())
    else:
        print(f"  {name.upper()} — not in df_scored (filtered out or no price history)")
    print()

# ── 4. Summary table ──────────────────────────────────────────────────────────
print("\n── Summary: wallet vs price-based scores ────────────────────────\n")
print(f"{'Market':<12} {'Label':<12} {'Wallet Score':<15} {'Price Score':<15} {'Agreement'}")
print("─" * 65)

for _, row in df_wallet_scores.iterrows():
    name = row["market"]
    keyword = market_name_map[name]
    matches = df_scored[df_scored["question"].str.contains(keyword, case=False, na=False)]
    price_score = f"{matches['suspicion_score'].max():.3f}" if len(matches) > 0 else "N/A"

    if price_score != "N/A":
        agreement = "✅ Both flag" if row["wallet_suspicion"] > 0.2 and float(price_score) > 0.02 else \
                    "⚠️  Wallet only" if row["wallet_suspicion"] > 0.2 else \
                    "⚠️  Price only" if float(price_score) > 0.02 else \
                    "❌ Neither flags"
    else:
        agreement = "⚠️  Wallet only (no price data)" if row["wallet_suspicion"] > 0.2 else "❌ Neither flags"

    print(f"{name:<12} {row['label']:<12} {row['wallet_suspicion']:<15.2f} {price_score:<15} {agreement}")

print("""
── Key thresholds used ───────────────────────────────────────────
  New wallet ratio 12h: >50% = high, >10% = elevated
  New wallet ratio 6h:  >10% = high, >5% = elevated
  Trade VPIN:           >0.80 = high, >0.60 = elevated
  Burst score:          >200 = high, >50 = elevated
  Directional consensus:>70% = high, >55% = elevated
""")

── Wallet suspicion scores (labeled markets) ────────────────────

  IRAN (CONFIRMED): wallet suspicion = 0.40

── Cross-checking against price-based model ─────────────────────

  IRAN — not in df_scored (filtered out or no price history)


── Summary: wallet vs price-based scores ────────────────────────

Market       Label        Wallet Score    Price Score     Agreement
─────────────────────────────────────────────────────────────────
iran         CONFIRMED    0.40            N/A             ⚠️  Wallet only (no price data)

── Key thresholds used ───────────────────────────────────────────
  New wallet ratio 12h: >50% = high, >10% = elevated
  New wallet ratio 6h:  >10% = high, >5% = elevated
  Trade VPIN:           >0.80 = high, >0.60 = elevated
  Burst score:          >200 = high, >50 = elevated
  Directional consensus:>70% = high, >55% = elevated



In [18]:
#@title 14 - Fetch wallet features for top 15 markets (aggregated Dune query)

top15 = df_scored.nlargest(15, "suspicion_score").copy().reset_index(drop=True)

def sql_quote(s):
    return "'" + s.replace("'", "''") + "'"

in_clause = ",\n    ".join(sql_quote(q) for q in top15["question"].tolist())

aggregated_sql = f"""
WITH trades AS (
    SELECT
        question,
        block_time,
        maker,
        price,
        amount,
        token_outcome_name
    FROM polymarket_polygon.market_trades
    WHERE question IN (
        {in_clause}
    )
),
market_stats AS (
    SELECT
        question,
        COUNT(*)                                          AS trade_count,
        COUNT(DISTINCT maker)                             AS unique_wallets,
        SUM(amount)                                       AS total_volume,
        MAX(block_time)                                   AS last_trade_time,
        MIN(block_time)                                   AS first_trade_time
    FROM trades
    GROUP BY question
),
wallet_volumes AS (
    SELECT question, maker, SUM(amount) AS wallet_volume
    FROM trades
    GROUP BY question, maker
),
top3 AS (
    SELECT question, SUM(wallet_volume) AS top3_volume
    FROM (
        SELECT question, wallet_volume,
               ROW_NUMBER() OVER (PARTITION BY question ORDER BY wallet_volume DESC) AS rn
        FROM wallet_volumes
    ) ranked
    WHERE rn <= 3
    GROUP BY question
),
resolution_times AS (
    SELECT question, MAX(block_time) AS inferred_resolution
    FROM trades
    GROUP BY question
),
new_wallets_12h AS (
    SELECT t.question, SUM(t.amount) AS new_wallet_volume_12h
    FROM trades t
    JOIN resolution_times r ON t.question = r.question
    WHERE t.maker IN (
        SELECT maker FROM (
            SELECT maker, question, MIN(block_time) AS first_seen
            FROM trades
            GROUP BY maker, question
        ) fw
        JOIN resolution_times rt ON fw.question = rt.question
        WHERE fw.first_seen >= rt.inferred_resolution - INTERVAL '12' HOUR
          AND fw.question = t.question
    )
    AND t.question = r.question
    GROUP BY t.question
),
new_wallets_6h AS (
    SELECT t.question, SUM(t.amount) AS new_wallet_volume_6h
    FROM trades t
    JOIN resolution_times r ON t.question = r.question
    WHERE t.maker IN (
        SELECT maker FROM (
            SELECT maker, question, MIN(block_time) AS first_seen
            FROM trades
            GROUP BY maker, question
        ) fw
        JOIN resolution_times rt ON fw.question = rt.question
        WHERE fw.first_seen >= rt.inferred_resolution - INTERVAL '6' HOUR
          AND fw.question = t.question
    )
    AND t.question = r.question
    GROUP BY t.question
),
burst AS (
    SELECT question, MAX(trades_in_hour) AS burst_score
    FROM (
        SELECT question, maker,
               DATE_TRUNC('hour', block_time) AS hour_bucket,
               COUNT(*) AS trades_in_hour
        FROM trades
        GROUP BY question, maker, DATE_TRUNC('hour', block_time)
    ) hourly
    GROUP BY question
),
directional AS (
    SELECT question,
           MAX(outcome_volume) * 1.0 / NULLIF(SUM(outcome_volume), 0) AS directional_consensus
    FROM (
        SELECT question, token_outcome_name, SUM(amount) AS outcome_volume
        FROM trades
        GROUP BY question, token_outcome_name
    ) ov
    GROUP BY question
),
vpin AS (
    SELECT question,
           ABS(yes_vol - no_vol) / NULLIF(yes_vol + no_vol, 0) AS trade_vpin
    FROM (
        SELECT question,
               SUM(CASE WHEN price > 0.5 THEN amount ELSE 0 END) AS yes_vol,
               SUM(CASE WHEN price <= 0.5 THEN amount ELSE 0 END) AS no_vol
        FROM trades
        GROUP BY question
    ) sides
)
SELECT
    ms.question,
    ms.trade_count,
    ms.unique_wallets,
    ms.total_volume,
    ms.first_trade_time,
    ms.last_trade_time,
    t3.top3_volume,
    t3.top3_volume / NULLIF(ms.total_volume, 0)        AS wallet_concentration,
    COALESCE(nw12.new_wallet_volume_12h, 0) / NULLIF(ms.total_volume, 0) AS new_wallet_ratio_12h,
    COALESCE(nw6.new_wallet_volume_6h, 0)  / NULLIF(ms.total_volume, 0) AS new_wallet_ratio_6h,
    b.burst_score,
    d.directional_consensus,
    v.trade_vpin
FROM market_stats ms
LEFT JOIN top3          t3  ON ms.question = t3.question
LEFT JOIN new_wallets_12h nw12 ON ms.question = nw12.question
LEFT JOIN new_wallets_6h  nw6  ON ms.question = nw6.question
LEFT JOIN burst           b   ON ms.question = b.question
LEFT JOIN directional     d   ON ms.question = d.question
LEFT JOIN vpin            v   ON ms.question = v.question
ORDER BY ms.question
"""

print("── Submitting aggregated Dune query ──────────────────────────────────")
print(f"  Markets: {len(top15)} | Returns: 1 row per market (no row limit issues)")
try:
    exec_id = execute_sql(aggregated_sql)
    print(f"  Execution ID: {exec_id}")
    success = poll_until_done(exec_id, timeout=180)
    if success:
        df_wallet_agg = fetch_results(exec_id)
        print(f"  ✅ Got {len(df_wallet_agg)} rows")
        print(df_wallet_agg[["question","trade_count","unique_wallets",
                              "wallet_concentration","new_wallet_ratio_12h",
                              "burst_score","trade_vpin"]].to_string())
    else:
        df_wallet_agg = pd.DataFrame()
except Exception as e:
    print(f"  ❌ Error: {e}")
    df_wallet_agg = pd.DataFrame()

# ── Score and combine ──────────────────────────────────────────────────────
if len(df_wallet_agg) > 0:
    print("\n── Combined scores ───────────────────────────────────────────────────")
    print(f"{'#':<3} {'Market':<50} {'Price':>7} {'Wallet':>7} {'Combined':>9}")
    print("─" * 80)

    combined = []
    for i, row in top15.iterrows():
        q           = row["question"]
        price_score = row["suspicion_score"]
        agg = df_wallet_agg[df_wallet_agg["question"] == q]

        if len(agg) > 0:
            a = agg.iloc[0]
            features = {
                "new_wallet_ratio":    float(a["new_wallet_ratio_12h"] or 0),
                "new_wallet_ratio_6h": float(a["new_wallet_ratio_6h"]  or 0),
                "trade_vpin":          float(a["trade_vpin"]            or 0),
                "burst_score":         float(a["burst_score"]           or 0),
                "directional_consensus": float(a["directional_consensus"] or 0),
            }
            wallet_score = compute_wallet_suspicion_score(features)
        else:
            wallet_score = None

        combined_score = (price_score + wallet_score) / 2 if wallet_score is not None else price_score
        combined.append({
            "question": q, "price_score": price_score,
            "wallet_score": wallet_score, "combined_score": combined_score
        })
        ws_str = f"{wallet_score:.3f}" if wallet_score is not None else "  N/A"
        print(f"{i+1:<3} {q[:50]:<50} {price_score:>7.3f} {ws_str:>7} {combined_score:>9.3f}")

    df_combined = pd.DataFrame(combined).sort_values("combined_score", ascending=False).reset_index(drop=True)
    print(f"\nTop 5 by combined score:")
    print(df_combined[["question","price_score","wallet_score","combined_score"]].head().to_string())

── Submitting aggregated Dune query ──────────────────────────────────
  Markets: 15 | Returns: 1 row per market (no row limit issues)
  Execution ID: 01KK7Z6013TG4F4V3BX2HJBB1Z
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_EXECUTING — waiting 5s...
  Status: QUERY_STATE_EXECUTING — waiting 5s...
  Status: QUERY_STATE_EXECUTING — waiting 5s...
  Status: QUERY_STATE_EXECUTING — 

In [19]:
#@title 15 - Save all data to Google Drive

import pickle
import os

SAVE_DIR = "/content/drive/MyDrive/insider_trading_poc"
os.makedirs(SAVE_DIR, exist_ok=True)

def save_checkpoint(obj, name):
    if obj is None:
        print(f"  ⚠️  Skipping {name} (is None)")
        return
    path = f"{SAVE_DIR}/{name}.pkl"
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    size_kb = os.path.getsize(path) / 1024
    print(f"  ✅ Saved {name} ({size_kb:.1f} KB)")

print("Saving to Google Drive...\n")

# Core model data
save_checkpoint(df_markets,   "df_markets")
save_checkpoint(df_scored,    "df_scored")
save_checkpoint(histories,    "histories")



# Wallet data
save_checkpoint(dune_results, "dune_results")
save_checkpoint(df_combined,  "df_combined")
save_checkpoint(df_wallet_agg, "df_wallet_agg")



# Cell 14 batch — only exists if Cell 14 ran successfully
try:
    save_checkpoint(df_batch, "df_batch")
except NameError:
    print("  ⚠️  Skipping df_batch (Cell 14 not yet run)")

print(f"\nAll data saved to {SAVE_DIR}")
print("Files saved:")
for f in os.listdir(SAVE_DIR):
    size_kb = os.path.getsize(f"{SAVE_DIR}/{f}") / 1024
    print(f"  {f} ({size_kb:.1f} KB)")

Saving to Google Drive...

  ✅ Saved df_markets (44.7 KB)
  ✅ Saved df_scored (18.7 KB)
  ✅ Saved histories (137.1 KB)
  ✅ Saved dune_results (11918.3 KB)
  ✅ Saved df_combined (1.8 KB)
  ✅ Saved df_wallet_agg (3.9 KB)
  ⚠️  Skipping df_batch (Cell 14 not yet run)

All data saved to /content/drive/MyDrive/insider_trading_poc
Files saved:
  df_wallet_agg.pkl (3.9 KB)
  dune_results.pkl (11918.3 KB)
  df_scored.pkl (18.7 KB)
  df_markets.pkl (44.7 KB)
  histories.pkl (137.1 KB)
  df_combined.pkl (1.8 KB)


In [20]:
#@title   16 — Export results to GitHub


from github import Github
from google.colab import userdata
import base64
import io

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_NAME = "chadallen/insider_trading_detection"
BRANCH = "main"

g = Github(GITHUB_TOKEN)
repo = g.get_repo(REPO_NAME)

def push_csv(df, filename, commit_message):
    """Push a dataframe as CSV to GitHub, creating or updating the file."""
    csv_content = df.to_csv(index=False)
    path = f"outputs/{filename}"

    try:
        # File exists — update it
        existing = repo.get_contents(path, ref=BRANCH)
        repo.update_file(path, commit_message, csv_content, existing.sha, branch=BRANCH)
        print(f"✅ Updated {path}")
    except Exception:
        # File doesn't exist — create it
        repo.create_file(path, commit_message, csv_content, branch=BRANCH)
        print(f"✅ Created {path}")


# Export each scored output
push_csv(df_scored,      "df_scored.csv",      "Update df_scored")
push_csv(df_wallet_agg,  "df_wallet_agg.csv",  "Update df_wallet_agg")
push_csv(df_combined,    "df_combined.csv",     "Update df_combined")
print("\nDone — results live at https://github.com/chadallen/insider_trading_detection/tree/main/outputs")

/tmp/ipykernel_151/3846627186.py:13: DeprecationWarning: Argument login_or_token is deprecated, please use auth=github.Auth.Token(...) instead
  g = Github(GITHUB_TOKEN)


✅ Updated outputs/df_scored.csv
✅ Updated outputs/df_wallet_agg.csv
✅ Updated outputs/df_combined.csv

Done — results live at https://github.com/chadallen/insider_trading_detection/tree/main/outputs


# Temp debugging cells


In [39]:
print(df_combined.head())
print(df_combined.columns.tolist())

                                            question  price_score  \
0  Will the Bitcoin Volatility Index hit 70 in 2026?     0.030003   
1       Rainbow FDV above $30M one day after launch?     0.004466   
2     Hyperlend FDV above $20M one day after launch?     0.043874   
3  Over $500K committed to the Sol Raffle public ...     0.033823   
4      Over $20M committed to the Space public sale?     0.024820   

   wallet_score  combined_score  
0           0.8        0.415002  
1           0.8        0.402233  
2           0.6        0.321937  
3           0.6        0.316911  
4           0.6        0.312410  
['question', 'price_score', 'wallet_score', 'combined_score']


In [45]:
print(df_scored.nlargest(5, 'suspicion_score')[['question', 'suspicion_score']].to_string())

                                              question  suspicion_score
109         Stable FDV above $2B one day after launch?         0.131836
33   Will Almanak FDV above $50M one day after launch?         0.108039
16      HumidiFi FDV above $300M one day after launch?         0.107422
52        Sentient FDV above $1B one day after launch?         0.102839
93      Sentient FDV above $800M one day after launch?         0.054056


In [46]:
print(df_wallet_agg['question'].tolist())
print("---")
print(df_scored['question'].head(5).tolist())

['Elsa FDV above $100M one day after launch?', 'HumidiFi FDV above $300M one day after launch?', 'Hyperlend FDV above $20M one day after launch?', 'Kinetiq FDV above $250M one day after launch?', 'Lighter FDV above $3B one day after launch?', 'Over $20M committed to the Space public sale?', 'Over $500K committed to the Sol Raffle public sale?', 'Rainbow FDV above $30M one day after launch?', 'Sentient FDV above $1B one day after launch?', 'Sentient FDV above $800M one day after launch?', 'Stable FDV above $2B one day after launch?', 'Tria FDV above $200M one day after launch?', 'Will Almanak FDV above $50M one day after launch?', 'Will Bitcoin dip to $65,000 by December 31, 2026?', 'Will the Bitcoin Volatility Index hit 70 in 2026?']
---
['Espresso FDV above $400M one day after launch?', 'Espresso FDV above $300M one day after launch?', 'Espresso FDV above $200M one day after launch?', 'Canton FDV above $3B one day after launch?', 'Will the Ethereum Volatility Index hit 80 in 2026?']
